# Pyr CA3 CAVE Setup Diagnostic

Diagnostic notebook for connecting directly to the pyr.ai / Zheng mouse hippocampus CA3 CAVE datastack, `zheng_ca3`, using a CAVE token loaded by the shared auth helper.

This notebook checks the local Python/CAVE environment, authenticates with CAVE, inspects datastack and volume metadata, verifies materialization versions and available tables, and includes optional CAVE query examples. Visualization and downstream analysis workflows are handled in subsequent notebooks.

## 1. Environment and Authentication

In [1]:
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

from caveclient import CAVEclient
import nglui


def find_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers and data."
    )


project_root = find_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from cave_auth import load_cave_token

print(f"Python: {sys.version.split()[0]}")
print(f"Python executable: {sys.executable}")
print(f"caveclient: {metadata.version('CAVEclient')}")
print(f"nglui: {metadata.version('nglui')}")
print(f"python-dotenv: {metadata.version('python-dotenv')}")

Python: 3.12.10
Python executable: d:\brainvolumes\.venv\Scripts\python.exe
caveclient: 8.0.1
nglui: 4.6.1
python-dotenv: 1.2.1


In [2]:
cave_token, cave_token_source = load_cave_token(project_root)

print("CAVE token loaded: True")
print(f"CAVE token source: {cave_token_source}")


CAVE token loaded: True
CAVE token source: sibling dotenv


## 2. CAVE Client and API Connectivity

In [3]:
def safe_call(label, fn, *args, **kwargs):
    """Run a diagnostic call and return (value, error_message)."""
    try:
        return fn(*args, **kwargs), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


datastack_name = "zheng_ca3"
client, client_error = safe_call(
    "CAVEclient",
    CAVEclient,
    datastack_name=datastack_name,
    auth_token=cave_token,
)

if client_error:
    print("Could not instantiate CAVEclient:")
    print(client_error)
else:
    print(f"Connected datastack: {client.datastack_name}")
    print(f"Effective CAVE server: {client.server_address}")
    print(f"Local service server: {getattr(client, 'local_server', None)}")

Connected datastack: zheng_ca3
Effective CAVE server: https://global.daf-apis.com
Local service server: https://minnie.microns-daf.com


## 3. Datastack and Volume Metadata

In [4]:
metadata_values = {}
metadata_errors = {}

if client is not None:
    checks = {
        "datastack_info": lambda: client.info.get_datastack_info(),
        "aligned_volume_info": lambda: client.info.get_aligned_volume_info(),
        "image_source": lambda: client.info.image_source(format_for="raw"),
        "segmentation_source": lambda: client.info.segmentation_source(format_for="raw"),
        "viewer_resolution": lambda: client.info.viewer_resolution(),
    }

    for label, fn in checks.items():
        value, err = safe_call(label, fn)
        metadata_values[label] = value
        metadata_errors[label] = err
else:
    metadata_errors["client"] = client_error


ds_info = metadata_values.get("datastack_info") if isinstance(metadata_values.get("datastack_info"), dict) else {}
av_info = metadata_values.get("aligned_volume_info") if isinstance(metadata_values.get("aligned_volume_info"), dict) else {}

authoritative_image_source = metadata_values.get("image_source") or av_info.get("image_source")
authoritative_segmentation_source = metadata_values.get("segmentation_source") or ds_info.get("segmentation_source")
viewer_resolution = metadata_values.get("viewer_resolution")
if hasattr(viewer_resolution, "tolist"):
    viewer_resolution = viewer_resolution.tolist()
elif viewer_resolution is not None:
    viewer_resolution = list(viewer_resolution)

ca3_info_rows = [
    ("datastack name", datastack_name if client is None else client.datastack_name),
    ("aligned volume", av_info.get("name") or (ds_info.get("aligned_volume") or {}).get("name")),
    ("image source", authoritative_image_source),
    ("segmentation source", authoritative_segmentation_source),
    ("synapse table", ds_info.get("synapse_table")),
    ("viewer resolution (discovered)", viewer_resolution),
    ("viewer resolution known Neuroglancer clue", [18, 18, 45]),
]

pd.DataFrame(ca3_info_rows, columns=["field", "value"])

,field,value
0,datastack name,zheng_ca3
1,aligned volume,zheng_ca3
2,image source,precomputed://gs://zheng_mouse_hippocampus_pro...
3,segmentation source,graphene://https://minnie.microns-daf.com/segm...
4,synapse table,synapses_ca3_v1
5,viewer resolution (discovered),"[18.0, 18.0, 45.0]"
6,viewer resolution known Neuroglancer clue,"[18, 18, 45]"


In [5]:
# Full CAVE datastack metadata, if exposed by the info service.
metadata_values.get("datastack_info")

{'aligned_volume': {'id': 12,
  'name': 'zheng_ca3',
  'display_name': 'Zheng CA3',
  'description': None,
  'image_source': 'precomputed://gs://zheng_mouse_hippocampus_production/v2/img_aligned_sharded_18nm'},
 'segmentation_source': 'graphene://https://minnie.microns-daf.com/segmentation/table/zheng_ca3',
 'skeleton_source': 'precomputed://middleauth+https://minnie.microns-daf.com/skeletoncache/api/v1/zheng_ca3/precomputed/skeleton',
 'analysis_database': None,
 'viewer_site': 'https://spelunker.cave-explorer.org/',
 'synapse_table': 'synapses_ca3_v1',
 'soma_table': None,
 'local_server': 'https://minnie.microns-daf.com',
 'description': None,
 'viewer_resolution_x': 18.0,
 'viewer_resolution_y': 18.0,
 'viewer_resolution_z': 45.0,
 'proofreading_status_table': None,
 'cell_identification_table': None,
 'proofreading_review_table': None}

In [6]:
# Full aligned-volume metadata, if exposed by the info service.
metadata_values.get("aligned_volume_info")

{'id': 12,
 'name': 'zheng_ca3',
 'display_name': 'Zheng CA3',
 'description': None,
 'image_source': 'precomputed://gs://zheng_mouse_hippocampus_production/v2/img_aligned_sharded_18nm'}

## 4. Materialization and Table Availability

In [7]:
versions = None
versions_error = None
most_recent_version = None
most_recent_timestamp = None
most_recent_timestamp_error = None

tables = None
tables_error = None

if client is not None:
    versions, versions_error = safe_call(
        "materialize.get_versions",
        lambda: client.materialize.get_versions(expired=True),
    )
    if versions_error:
        print("materialize.get_versions(expired=True) failed:")
        print(versions_error)
    else:
        versions = sorted(versions)
        print(f"Available materialization versions: {len(versions)} total")
        if versions:
            print(f"First versions: {versions[:5]}")
            print(f"Recent versions: {versions[-10:]}")
            most_recent_version = max(versions)
            most_recent_timestamp, most_recent_timestamp_error = safe_call(
                "materialize.get_timestamp",
                lambda: client.materialize.get_timestamp(most_recent_version),
            )
            print(f"Most recent materialization version: {most_recent_version}")
            if most_recent_timestamp_error:
                print("materialize.get_timestamp failed:")
                print(most_recent_timestamp_error)
            else:
                print(f"Most recent materialization timestamp: {most_recent_timestamp}")

    tables, tables_error = safe_call(
        "materialize.get_tables",
        lambda: client.materialize.get_tables(),
    )
    if tables_error:
        print("materialize.get_tables() failed:")
        print(tables_error)
    else:
        tables = sorted(tables)
else:
    print("Skipping materialization diagnostics because CAVEclient was not created.")

Available materialization versions: 706 total
First versions: [1, 2, 3, 4, 5]
Recent versions: [695, 696, 697, 698, 699, 700, 701, 702, 703, 704]
Most recent materialization version: 704
Most recent materialization timestamp: 2026-08-28 13:11:34.685303+00:00


In [8]:
pd.DataFrame({"materialization_table": tables or []})

,materialization_table
0,c3_nuclei_v1
1,ca3_cell_id
2,ca3_cell_type
3,synapses_ca3_v1


### API Call Status

In [9]:
status_rows = [
    {"check": "client connection", "ok": client_error is None, "error": client_error},
]

metadata_check_labels = {
    "datastack_info": "datastack metadata",
    "aligned_volume_info": "aligned-volume metadata",
    "image_source": "image source",
    "segmentation_source": "segmentation source",
    "viewer_resolution": "viewer resolution",
}

for label, err in metadata_errors.items():
    status_rows.append(
        {
            "check": metadata_check_labels.get(label, label),
            "ok": err is None,
            "error": err,
        }
    )

timestamp_status_error = most_recent_timestamp_error
if most_recent_timestamp is None and timestamp_status_error is None:
    timestamp_status_error = "Timestamp not checked or unavailable"

status_rows.extend(
    [
        {"check": "materialization versions", "ok": versions_error is None, "error": versions_error},
        {"check": "latest materialization timestamp", "ok": most_recent_timestamp is not None and timestamp_status_error is None, "error": timestamp_status_error},
        {"check": "materialization tables", "ok": tables_error is None, "error": tables_error},
    ]
)

status_df = pd.DataFrame(status_rows)
if status_df["error"].notna().any():
    display(status_df)
else:
    display(status_df.drop(columns=["error"]))

,check,ok
0,client connection,True
1,datastack metadata,True
2,aligned-volume metadata,True
3,image source,True
4,segmentation source,True
5,viewer resolution,True
6,materialization versions,True
7,latest materialization timestamp,True
8,materialization tables,True


## 5. Optional CAVE Query Examples

In [10]:
example_tables = [
    "ca3_cell_id",
    "ca3_cell_type",
    "c3_nuclei_v1",
    "synapses_ca3_v1",
]

optional_versions, optional_versions_error = safe_call(
    "materialize.get_versions",
    lambda: client.materialize.get_versions(),
)

count_rows = []

if optional_versions_error or not optional_versions:
    print("Optional CAVE queries skipped: materialization versions unavailable.")
else:
    for version in optional_versions:
        for table in example_tables:
            try:
                n = client.materialize.get_annotation_count(
                    table,
                    version=version
                )
                count_rows.append(
                    {
                        "version": version,
                        "table": table,
                        "ok": True,
                        "count": n,
                        "error_type": None,
                        "error_summary": None,
                    }
                )
            except Exception as e:
                error_summary = str(e).split(" for url:")[0]
                count_rows.append(
                    {
                        "version": version,
                        "table": table,
                        "ok": False,
                        "count": None,
                        "error_type": type(e).__name__,
                        "error_summary": error_summary,
                    }
                )

    availability_df = pd.DataFrame(count_rows)
    availability_display = availability_df.copy()
    availability_display["row_count"] = availability_display["count"].astype("Int64")
    availability_display["availability"] = availability_display["ok"].map({True: "available", False: "unavailable"})
    availability_display["count_or_status"] = availability_display.apply(
        lambda row: int(row["row_count"]) if row["ok"] else "unavailable",
        axis=1,
    )

    availability_matrix = availability_display.pivot(
        index="version",
        columns="table",
        values="count_or_status",
    ).reset_index()

    unavailable_rows = availability_display.loc[
        ~availability_display["ok"],
        ["version", "table", "availability", "error_type"],
    ]

    display(availability_matrix)

    if not unavailable_rows.empty:
        display(unavailable_rows)


table,version,c3_nuclei_v1,ca3_cell_id,ca3_cell_type,synapses_ca3_v1
0,1,unavailable,unavailable,unavailable,36834775
1,195,35499,1,1,36834775
2,357,35499,1,1,36834775
3,681,35499,1,1,36834775
4,695,35499,1,1,36834775
5,703,35499,1,1,36834775
6,704,35499,1,1,36834775


,version,table,availability,error_type
0,1,ca3_cell_id,unavailable,HTTPError
1,1,ca3_cell_type,unavailable,HTTPError
2,1,c3_nuclei_v1,unavailable,HTTPError


In [11]:
materialization_version = 195

ca3_cell_id_df = client.materialize.query_table(
    "ca3_cell_id",
    materialization_version=materialization_version
)

ca3_cell_type_df = client.materialize.query_table(
    "ca3_cell_type",
    materialization_version=materialization_version
)

c3_nuclei_df = client.materialize.query_table(
    "c3_nuclei_v1",
    materialization_version=materialization_version
)

print("ca3_cell_id:", ca3_cell_id_df.shape)
print("ca3_cell_type:", ca3_cell_type_df.shape)
print("c3_nuclei_v1:", c3_nuclei_df.shape)

ca3_cell_id: (1, 10)
ca3_cell_type: (1, 9)
c3_nuclei_v1: (35499, 10)


In [12]:
metadata_fields = [
    "id",
    "schema_type",
    "valid",
    "read_permission",
    "write_permission",
    "created",
    "last_updated",
    "description",
]

for table in ["ca3_cell_id", "ca3_cell_type", "c3_nuclei_v1"]:
    print(f"\n{'='*70}")
    print(table)
    print("="*70)

    meta = client.annotation.get_table_metadata(table)
    meta_summary = {key: meta.get(key) for key in metadata_fields if key in meta}
    display(pd.DataFrame([meta_summary]).T.rename(columns={0: "value"}))

    df = client.materialize.query_table(
        table,
        materialization_version=materialization_version
    )

    print("Columns:")
    print(df.columns.tolist())
    print("Shape:", df.shape)

    if len(df) <= 5:
        display(df)
    else:
        display(df.head(5))



ca3_cell_id


,value
id,3
schema_type,nucleus_detection
valid,True
read_permission,PUBLIC
write_permission,PRIVATE
created,2025-01-07T18:06:36.663345
last_updated,2026-08-28T20:31:50.701146
description,


Columns:
['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_supervoxel_id', 'pt_root_id', 'pt_position', 'bb_start_position', 'bb_end_position']
Shape: (1, 10)


,id,created,superceded_id,valid,volume,pt_supervoxel_id,pt_root_id,pt_position,bb_start_position,bb_end_position
0,1,2025-01-07 18:06:35.764595+00:00,<NA>,True,<NA>,74314342124163167,648518346440912934,"[37360, 57056, 2063]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"



ca3_cell_type


...

,id,created,superceded_id,valid,classification_system,cell_type,pt_supervoxel_id,pt_root_id,pt_position
0,1,2025-01-07 16:45:38.990575+00:00,<NA>,True,ca3_main,pyr,74314342124163167,648518346440912934,"[37360, 57056, 2063]"



c3_nuclei_v1


,value
id,4
schema_type,nucleus_detection
valid,True
read_permission,PUBLIC
write_permission,GROUP
created,2025-02-20T18:43:05.720385
last_updated,2026-08-28T20:31:50.701146
description,Nuclei locations and volumes from nucleus dete...


Columns:
['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_supervoxel_id', 'pt_root_id', 'pt_position', 'bb_start_position', 'bb_end_position']
Shape: (35499, 10)


,id,created,superceded_id,valid,volume,pt_supervoxel_id,pt_root_id,pt_position,bb_start_position,bb_end_position
0,20099,2025-02-20 18:43:53.115880+00:00,<NA>,True,0.104509,76567172461640457,648518346433938771,"[53360, 64608, 1103]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
1,23059,2025-02-20 18:43:55.053192+00:00,<NA>,True,2.642596,77131084286202290,648518346436811563,"[57632, 71840, 281]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
2,24798,2025-02-20 18:43:56.143105+00:00,<NA>,True,0.167962,77623184861837124,648518346438478107,"[61152, 68272, 1809]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
3,13183,2025-02-20 18:43:22.039071+00:00,<NA>,True,264.815735,75088054712378344,648518346450796819,"[42848, 54624, 1911]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
4,16387,2025-02-20 18:43:40.930762+00:00,<NA>,True,0.126904,75652104110407854,648518346451269875,"[47088, 62528, 1683]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"


In [13]:
ca3_cell_id_df = client.materialize.query_table("ca3_cell_id")
ca3_cell_type_df = client.materialize.query_table("ca3_cell_type")
c3_nuclei_df = client.materialize.query_table("c3_nuclei_v1")

In [14]:
def display_table_preview(name, df, max_full_rows=5, preview_rows=5):
    print(f"{name}: shape {df.shape}")
    print("Columns:")
    print(df.columns.tolist())

    if len(df) <= max_full_rows:
        display(df)
    else:
        display(df.head(preview_rows))


display_table_preview("ca3_cell_id", ca3_cell_id_df)
display_table_preview("ca3_cell_type", ca3_cell_type_df)
display_table_preview("c3_nuclei_v1", c3_nuclei_df)


ca3_cell_id: shape (1, 10)
Columns:
['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_supervoxel_id', 'pt_root_id', 'pt_position', 'bb_start_position', 'bb_end_position']


,id,created,superceded_id,valid,volume,pt_supervoxel_id,pt_root_id,pt_position,bb_start_position,bb_end_position
0,1,2025-01-07 18:06:35.764595+00:00,<NA>,True,<NA>,74314342124163167,648518346448625630,"[37360, 57056, 2063]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"


ca3_cell_type: shape (1, 9)
Columns:
['id', 'created', 'superceded_id', 'valid', 'classification_system', 'cell_type', 'pt_supervoxel_id', 'pt_root_id', 'pt_position']


,id,created,superceded_id,valid,classification_system,cell_type,pt_supervoxel_id,pt_root_id,pt_position
0,1,2025-01-07 16:45:38.990575+00:00,<NA>,True,ca3_main,pyr,74314342124163167,648518346448625630,"[37360, 57056, 2063]"


c3_nuclei_v1: shape (35499, 10)
Columns:
['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_supervoxel_id', 'pt_root_id', 'pt_position', 'bb_start_position', 'bb_end_position']


,id,created,superceded_id,valid,volume,pt_supervoxel_id,pt_root_id,pt_position,bb_start_position,bb_end_position
0,20099,2025-02-20 18:43:53.115880+00:00,<NA>,True,0.104509,76567172461640457,648518346433938771,"[53360, 64608, 1103]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
1,23059,2025-02-20 18:43:55.053192+00:00,<NA>,True,2.642596,77131084286202290,648518346436811563,"[57632, 71840, 281]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
2,12179,2025-02-20 18:43:21.413910+00:00,<NA>,True,0.966712,75019678833055912,648518346443111625,"[42352, 69360, 1920]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
3,12184,2025-02-20 18:43:21.416918+00:00,<NA>,True,3.609308,75019678833103367,648518346443111625,"[42320, 69312, 1952]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"
4,11192,2025-02-20 18:43:20.796546+00:00,<NA>,True,0.126904,75015830274180832,648518346442589323,"[42256, 40784, 1023]","[<NA>, <NA>, <NA>]","[<NA>, <NA>, <NA>]"


In [15]:
sorted_c3_nuclei_df = c3_nuclei_df.sort_values(by="volume")
nuclei_preview_columns = ["id", "volume", "pt_root_id", "pt_position"]

print("c3_nuclei_v1 sorted by volume:", sorted_c3_nuclei_df.shape)
print("Smallest volumes:")
display(sorted_c3_nuclei_df.loc[:, nuclei_preview_columns].head(5))
print("Largest volumes:")
display(sorted_c3_nuclei_df.loc[:, nuclei_preview_columns].tail(5))

c3_nuclei_v1 sorted by volume: (35499, 10)
Smallest volumes:


,id,volume,pt_root_id,pt_position
8388,16545,0.003732,648518346445658400,"[48256, 68512, 701]"
22883,35319,0.011197,0,"[71904, 40192, 1926]"
23565,7608,0.011197,648518346446223870,"[35568, 54080, 1802]"
1571,4471,0.011197,648518346437360248,"[29824, 73248, 1014]"
31495,21066,0.011197,648518346435539098,"[56288, 53504, 1320]"


Largest volumes:


,id,volume,pt_root_id,pt_position
32399,21199,1593.985474,648518346446374761,"[56912, 58608, 1638]"
677,20043,1597.512695,648518346455289164,"[55104, 58736, 1789]"
996,17487,1598.233032,648518346445705245,"[49472, 52800, 1809]"
1524,22876,1677.171265,648518346449558821,"[57840, 61168, 1754]"
24376,27858,2389.182861,0,"[65696, 32416, 1448]"


## 6. Summary

This notebook confirms the local Python/CAVE environment and authentication, connects to `zheng_ca3`, inspects datastack and volume metadata, checks materialization versions and available tables, and demonstrates optional example CAVE queries. Visualization and downstream workflows are handled in subsequent notebooks.